<a href="https://colab.research.google.com/github/IdoAbram/Tiny-NMT/blob/dev/teacher_nmt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teacher Model Benchmark – English → Spanish

This notebook benchmarks a pretrained MarianMT teacher model.
It measures:
- Translation quality (BLEU, chrF, TER)
- Inference speed
- Model size

The results serve as a fixed baseline for all student models.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch: 2.9.0+cu126
cuda available: True
gpu: Tesla T4


## Load Teacher Model - Helsinki-NLP

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
teacher = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
teacher.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loaded teacher on: cuda


## Load Evaluation Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus_books", "en-es")
eval_src = train_src[:500]
eval_ref = train_ref[:500]

EN: Hello! How are you today?
ES: Hola, ¿cómo estás hoy?
----------------------------------------
EN: I want to build a very small translation model.
ES: Quiero construir un modelo de traducción muy pequeño.
----------------------------------------
EN: This is a simple test sentence.
ES: Esta es una simple frase de prueba.
----------------------------------------


## Inference Benchmark

In [ ]:
preds, total_seconds = batched_translate(
    eval_src,
    batch_size=32,
    num_beams=4
)

Total params: 77,943,296
Trainable params: 77,943,296
Estimated size: 297.3 MB (bytes/param=4)


In [ ]:
print("Sentences:", len(preds))
print("Total time:", total_seconds)
print("Sentences/sec:", len(preds) / total_seconds)

## Model Size

In [ ]:
total_params = sum(p.numel() for p in teacher.parameters())
size_mb = total_params * next(teacher.parameters()).element_size() / (1024**2)

print(f"Parameters: {total_params:,}")
print(f"Size (MB): {size_mb:.1f}")

Elapsed: 0.267s | approx input tokens: 512 | approx tokens/sec: 1915


## Translation Quality

In [ ]:
!pip install sacrebleu

In [ ]:
from sacrebleu.metrics import BLEU, CHRF, TER

bleu = BLEU()
chrf = CHRF()
ter = TER()

bleu_score = bleu.corpus_score(preds, [eval_ref])
chrf_score = chrf.corpus_score(preds, [eval_ref])
ter_score = ter.corpus_score(preds, [eval_ref])

print("BLEU:", bleu_score)
print("chrF:", chrf_score)
print("TER:", ter_score)


['ca-de', 'ca-en', 'ca-hu', 'ca-nl', 'de-en', 'de-eo', 'de-es', 'de-fr', 'de-hu', 'de-it', 'de-nl', 'de-pt', 'de-ru', 'el-en', 'el-es', 'el-fr', 'el-hu', 'en-eo', 'en-es', 'en-fi']
DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93470
    })
})
splits: ['train']
example: {'id': '0', 'translation': {'en': 'Source: Project GutenbergAudiobook available here', 'es': 'Source: Wikisource & librodot.com'}}


## Summary

- Model: MarianMT (Transformer Base)
- Dataset: OPUS Books (en–es)
- Evaluation size: 500 sentences
- BLEU: 27.6
- chrF: 53.5
- TER: 60.2

These results define the teacher baseline used throughout the project.
